# v5r 消融實驗 — 階段 2 / 階段 3 共用

同一份 notebook 跑六個消融臂與最後的長跑。**只需要編輯下一個 cell 的 `ARM` 與 `EPOCHS`**，
其餘全部由 `v9_modules.py` 的 `ARMS` 表推導，沒有第二個地方需要同步修改。

| 臂 | 與 A0 的唯一差異 | imgsz | box | 預估時數 (60 輪) |
| --- | --- | ---: | ---: | ---: |
| **A0** | —（官方 yolo26-p2 + 內建 CIoU） | 640 | 8.0 | 2.75 h |
| **A4** | `imgsz = 800` | 800 | 8.0 | 4.50 h |
| **A2** | stride-2 Conv → ADown ×6 | 640 | 8.0 | 2.67 h |
| **A1** | CIoU → Wise-Inner-MPDIoU (ratio=0.70) | 640 | 3.70 | 2.92 h |
| **A1b** | 同 A1 但 ratio=1.25 | 640 | 4.15 | 2.92 h |
| **A3** | C3k2 → StarTripletBlock ×3 | 640 | 8.0 | 6.00 h |

**建議順序 A0 → A4 → A2 → A1 → A1b → A3**。前四臂 12.83 h 就能回答最關鍵的問題
（解析度值不值得、ADown 有沒有用、loss 有沒有用），而且不必先決定 A3 的去留。
六臂合計 21.75 h，在 Kaggle 單週 30 h 配額內。

A1 與 A1b 的 `box` 是校準過的：自訂損失的量級隨 ratio 而不同
（ratio=0.70 為 CIoU 的 2.17x、ratio=1.25 為 1.93x），`box = 8.0 / 倍率`
讓兩臂的等效權重都對齊 A0 的 8.0，測的是損失的形狀而非量級。

### 執行前確認
1. `Datasets_YOLO26_v5r/OutPut/` 已上傳成 Kaggle Dataset，路徑填進下方 `DATASET_SRC`。
2. **Notebook 設定裡的 Internet 必須開啟** —— 要抓 `v9_modules.py` 與 `yolo26n.pt`。
3. 建議用 **Save & Run All**，全程無人看顧。

### 判準（詳見 `docs/v9_實驗重整計畫.md` 階段 1）
- **基準**：A0 第 45–60 輪的平台期平均，**不是** v8 的歷史值 —— 標註已改，量尺不同。
- **σ**：用 A0 自己第 45–60 輪的標準差重新校準。v5r 的 valid Thrips 實例數只剩 125
  （v5 是 230），per-class AP 解析度變粗，舊的 σ=0.0069 已不適用。
- **主判準** mAP@0.5:0.95 平台期平均，需超出 A0 達 **2σ** 才納入。
- **次判準** Thrips AP@0.5、Scale_Insect FP 數、s/epoch。
- **A3 與 A4 記憶體互斥**，兩者都過門檻時取增益大者；相當時取 A4。

In [ ]:
# ══════════════════════════════════════════════════════════════════════
# 唯一需要編輯的 cell
# ══════════════════════════════════════════════════════════════════════
ARM    = "A0"       # A0 / A4 / A2 / A1 / A1b / A3
EPOCHS = 60         # 消融 60；階段 3 長跑改 160

# Datasets_YOLO26_v5r/OutPut 上傳成 Kaggle Dataset 之後的路徑
DATASET_SRC = "/kaggle/input/datasets-yolo26-v5r"

# 長跑用的安全閥。EPOCHS=60 時不會觸發，等於免費的當機保險。
STOP_AFTER_EPOCHS = 105     # 跑滿這麼多輪就乾淨停止，剩下的交給 RESUME
DEADLINE_HOURS    = 10.5    # Save & Run All 上限 12 h，留 1.5 h 餘裕

# Step.1 環境

In [ ]:
!nvidia-smi
# 版本必須釘死：整套注入機制建立在 8.4.121 的 parse_model 與 BboxLoss 實作細節上
!pip install -q ultralytics==8.4.121

import ultralytics
assert ultralytics.__version__ == "8.4.121", \
    f"ultralytics 版本不符：{ultralytics.__version__}，注入機制可能失效"
print(f"▷ ultralytics {ultralytics.__version__}")

In [ ]:
# 取得 v9_modules.py —— 自訂 loss、自訂模組、六臂定義的單一真實來源。
# 直接抓 GitHub 上的最新版，避免 notebook 內再出現一份複製貼上（問題 C3）。
import subprocess, sys

URL = ("https://raw.githubusercontent.com/DreamOver9183/"
       "AY2026_Citrus_Pests_and_Diseases_Project/main/Train%20Code/v9/v9_modules.py")
subprocess.run(["curl", "-sSLf", "-o", "/kaggle/working/v9_modules.py", URL], check=True)

sys.path.insert(0, "/kaggle/working")
import v9_modules as v9

assert v9.REQUIRED_ULTRALYTICS == ultralytics.__version__, \
    f"v9_modules 要求 ultralytics {v9.REQUIRED_ULTRALYTICS}"
assert ARM in v9.ARMS, f"未知的臂 {ARM}，可用：{list(v9.ARMS)}"
print(f"▷ v9_modules {v9.__version__}\n")
print(v9.arm_summary(ARM, EPOCHS))

# Step.2 資料集準備
### 複製到可寫入工作區、校驗完整性、清 BOM 與舊快取、改寫 data.yaml 路徑。

In [ ]:
import os, shutil, sys, yaml

def render_progress_bar(current, total, task_name="檔案同步複製中", bar_length=25):
    percent = (current / total) * 100 if total > 0 else 100.0
    filled = int(bar_length * current // total) if total > 0 else bar_length
    bar = "█" * filled + "░" * (bar_length - filled)
    sys.stdout.write(f"\r▷ 正在執行 [{task_name}] | 進度: [{bar}] {percent:5.1f}% ({current}/{total})")
    sys.stdout.flush()


def copy_and_verify_dataset(src_dir, dst_dir):
    if not os.path.exists(src_dir):
        print(f"▷ 錯誤：找不到來源資料集目錄 {src_dir}")
        return False

    src_files = []
    for root, _, files in os.walk(src_dir):
        for file in files:
            src_files.append(os.path.relpath(os.path.join(root, file), src_dir))
    total_files = len(src_files)
    print(f"▷ 來源資料集掃描完成，共計 {total_files} 個檔案")

    for idx, rel_path in enumerate(src_files, 1):
        dst_path = os.path.join(dst_dir, rel_path)
        os.makedirs(os.path.dirname(dst_path), exist_ok=True)
        shutil.copy2(os.path.join(src_dir, rel_path), dst_path)
        if idx % 200 == 0 or idx == total_files:
            render_progress_bar(idx, total_files)

    print("\n\n▷ 正在檢查複製檔案")
    dst_files_set = set()
    for root, _, files in os.walk(dst_dir):
        for file in files:
            dst_files_set.add(os.path.relpath(os.path.join(root, file), dst_dir))

    missing, corrupted = [], []
    for rel_path in src_files:
        if rel_path not in dst_files_set:
            missing.append(rel_path)
        elif os.path.getsize(os.path.join(src_dir, rel_path)) != \
                os.path.getsize(os.path.join(dst_dir, rel_path)):
            corrupted.append(rel_path)

    print("≡" * 60)
    print("▷ 資料集複製完整性校驗：")
    print(f"  ▶ 來源檔案總數 : {len(src_files)}")
    print(f"  ▶ 目標檔案總數 : {len(dst_files_set)}")
    print(f"  ▶ 遺漏檔案數   : {len(missing)}")
    print(f"  ▶ 損毀/大小不符: {len(corrupted)}")
    ok = not missing and not corrupted
    print("▷ 檢查通過" if ok else f"▷ 檢查失敗  遺漏={missing[:5]}  損毀={corrupted[:5]}")
    print("≡" * 60)
    return ok


DST = "/kaggle/working/datasets-yolo26-v5r"
DATA_YAML = "/kaggle/working/data.yaml"

assert copy_and_verify_dataset(DATASET_SRC, DST), "資料集複製失敗，不要往下跑"

# data.yaml 改寫成絕對路徑（來源版本用的是相對的 path: .）
with open(os.path.join(DST, "data.yaml"), encoding="utf-8") as f:
    ycfg = yaml.safe_load(f)
ycfg.update(path=DST, train="train/images", val="valid/images", test="test/images")
with open(DATA_YAML, "w", encoding="utf-8") as f:
    yaml.safe_dump(ycfg, f, default_flow_style=False, allow_unicode=True)
assert ycfg["nc"] == 8, f"nc={ycfg['nc']}，應為 8"

# BOM 與舊快取會讓 Ultralytics 的標註解析出錯
bom_fixed = cache_removed = 0
for subdir, _, files in os.walk(DST):
    for file in files:
        path = os.path.join(subdir, file)
        if file.endswith(".cache"):
            os.remove(path)
            cache_removed += 1
        elif file.endswith(".txt") and "labels" in subdir:
            with open(path, "rb") as fh:
                is_bom = fh.read(3) == b"\xef\xbb\xbf"
            if is_bom:
                with open(path, encoding="utf-8-sig") as fh:
                    content = fh.read()
                with open(path, "w", encoding="utf-8") as fh:
                    fh.write(content)
                bom_fixed += 1

print(f"▷ data.yaml → {DATA_YAML}   nc={ycfg['nc']}")
print(f"▷ names = {ycfg['names']}")
print(f"▷ 修正 BOM {bom_fixed} 個、清除快取 {cache_removed} 個")
print("▷ Step.2 完成")

# Step.3 建構本臂的模型組態

In [ ]:
import yaml as _yaml
from ultralytics.nn.tasks import DetectionModel
from ultralytics.utils.torch_utils import get_flops, get_num_params

# 依 ARMS 表裝好該臂需要的 loss / 模組，並取得超參數
HP = v9.install_arm(ARM, epochs=EPOCHS)

cfg = v9.make_arm_yaml(nc=ycfg["nc"], **v9.ARMS[ARM]["arch"])
# 檔名必須帶 scale 字母：yaml_model_load 會用檔名覆寫 dict 裡的 scale 鍵，
# 少了 "n" 會落回 scales 字典的第一項並印出 warning —— 目前碰巧正確，但很脆弱
YAML_PATH = f"/kaggle/working/yolo26n-p2-{ARM}.yaml"
with open(YAML_PATH, "w", encoding="utf-8") as f:
    _yaml.safe_dump(cfg, f, sort_keys=False, allow_unicode=True)

_m = DetectionModel(cfg=cfg, ch=3, nc=ycfg["nc"], verbose=False)
print(f"▷ {ARM}: {get_num_params(_m):,} params / {get_flops(_m, imgsz=640):.2f} GFLOPs @640")
print(f"▷ yaml → {YAML_PATH}")
print("▷ 超參數：" + "  ".join(f"{k}={v}" for k, v in sorted(HP.items())))
del _m

# Step.4 架構隔離驗證
### 在燒 GPU 時數之前確認：體系正確、模組就位、可前向、損失有限。

In [ ]:
import math, torch
from ultralytics import YOLO
from ultralytics.cfg import get_cfg
import ultralytics.utils.loss

model = YOLO(YAML_PATH)
core = model.model
spec = v9.ARMS[ARM]

# 1. E2E 體系必須與 v8 一致，否則走的是另一個損失路徑、完全無法對照
det = core.model[-1]
assert getattr(det, "end2end", False), "Detect 不是 end2end：yaml 少了 end2end: True"
assert det.reg_max == 1, f"reg_max={det.reg_max}，應為 1"
assert det.nc == ycfg["nc"] and det.nl == 4, f"nc={det.nc} nl={det.nl}"
strides = [int(s) for s in det.stride]
assert strides == [4, 8, 16, 32], f"strides={strides}"
print(f"▷ 1/4 end2end=True, reg_max=1, nc={det.nc}, 偵測頭={det.nl}, strides={strides}")

# 2. 該臂該有的模組就位、不該有的不能出現（避免前一臂的注入殘留）
types = {i: m.type.split(".")[-1] for i, m in enumerate(core.model)}
want_adown = [i for i, _ in v9.ADOWN_LAYERS] if spec["arch"]["adown"] else []
want_stb = [i for i, _ in v9.STB_LAYERS] if spec["arch"]["stb"] else []
for i in want_adown:
    assert types[i] == "ADown", f"第 {i} 層是 {types[i]}，應為 ADown"
for i in want_stb:
    assert types[i] == "StarTripletBlock", f"第 {i} 層是 {types[i]}，別名注入沒生效"
if not spec["arch"]["stb"]:
    assert "StarTripletBlock" not in types.values(), "不該出現 StarTripletBlock"
patched = ultralytics.utils.loss.bbox_iou.__name__ == "bbox_wise_inner_mpdiou"
assert patched == (spec["loss_ratio"] is not None), f"自訂 loss 安裝狀態不符 ({patched})"
print(f"▷ 2/4 ADown@{want_adown or '—'}  STB@{want_stb or '—'}  "
      f"loss={'自訂 ratio=' + str(spec['loss_ratio']) if patched else '內建 CIoU'}")

# 3. 前向
core.eval()
with torch.no_grad():
    core(torch.zeros(1, 3, HP["imgsz"], HP["imgsz"]))
print(f"▷ 3/4 Forward pass ({HP['imgsz']}x{HP['imgsz']}) 成功")

# 4. 完整損失路徑，刻意用微小框貼近本資料集分佈
core.args = get_cfg(overrides={"box": HP["box"], "cls": HP["cls"], "dfl": HP["dfl"]})
core.train()
loss, items = core.loss({
    "img": torch.rand(2, 3, HP["imgsz"], HP["imgsz"]),
    "batch_idx": torch.tensor([0.0, 0.0, 1.0]),
    "cls": torch.tensor([[4.0], [6.0], [1.0]]),      # Scale_Insect / Thrips / Canker
    "bboxes": torch.tensor([[0.50, 0.50, 0.04, 0.04],
                            [0.22, 0.31, 0.02, 0.03],
                            [0.71, 0.68, 0.15, 0.12]]),
})
vals = ({k: float(v) for k, v in items.items()} if isinstance(items, dict)
        else {k: float(v) for k, v in zip(("box", "cls", "l1"), items.flatten())})
assert torch.isfinite(loss).all() and all(math.isfinite(v) for v in vals.values()), vals
print("▷ 4/4 損失路徑通過   " + "  ".join(f"{k}={v:.4f}" for k, v in vals.items()))
print("\n▷ Step.4 全部通過，可以開始訓練")

# Step.5 訓練

In [ ]:
import shutil, time
from ultralytics import YOLO

model = YOLO(YAML_PATH)
model.load("yolo26n.pt")     # 轉移率低是預期的：被換掉的層只能從零開始學


def stop_and_snapshot(trainer):
    """每輪保留可續跑的 checkpoint，並在時數/輪數上限時乾淨停止。

    訓練迴圈結束後一定會執行 final_eval() → strip_optimizer()，把 last.pt / best.pt
    的 epoch 改成 -1 並清掉 optimizer/EMA，那種檔案無法續跑。
    on_fit_epoch_end 的觸發點在 save_model() 之後、跳出迴圈之前
    （trainer.py 的 610 / 624 / 633 行），此時 last.pt 才剛寫好且尚未被 strip。

    備份不設條件：patience 早停時 trainer.stop 在 trainer.py:605 就已為 True，
    若寫成 `if not trainer.stop` 這段會整個被跳過 —— 那正是原本的失效模式。
    """
    if trainer.last.exists():
        shutil.copy(trainer.last, trainer.wdir / "resume_from.pt")
    # 自訂 loss 的離群度統計量不在 checkpoint 裡，續跑要靠這份側錄還原
    if ultralytics.utils.loss.bbox_iou.__name__ == "bbox_wise_inner_mpdiou":
        (trainer.wdir / "wiou_state.txt").write_text(str(v9.wiou_state()))

    elapsed = (time.time() - trainer.train_time_start) / 3600
    if not trainer.stop and (trainer.epoch + 1 >= STOP_AFTER_EPOCHS
                             or elapsed > DEADLINE_HOURS):
        trainer.stop = True
        print(f"\n▷ 停於第 {trainer.epoch + 1} / {trainer.epochs} 輪，已耗時 {elapsed:.2f} h")
        print(f"▷ 續跑用 checkpoint：{trainer.wdir / 'resume_from.pt'}")


model.add_callback("on_fit_epoch_end", stop_and_snapshot)

results = model.train(data=DATA_YAML, save_period=10, **HP)
print(f"▷ {ARM} 訓練完畢")

# Step.6 輸出整理

In [ ]:
import os, shutil

runs_dir = "/kaggle/working/runs"
if os.path.exists(runs_dir):
    print("▷ 正在壓縮訓練輸出")
    shutil.make_archive(f"/kaggle/working/runs_{ARM}", "zip", runs_dir)
    size = os.path.getsize(f"/kaggle/working/runs_{ARM}.zip") / (1024 ** 2)
    print(f"▷ 壓縮成功 /kaggle/working/runs_{ARM}.zip ({size:.2f} MB)")
    print("\n▷ 下載後把 ARM 改成下一個臂再跑一次。")
    print("▷ 順序：A0 → A4 → A2 → A1 → A1b → A3")
else:
    print(f"▷ 壓縮失敗：找不到 {runs_dir}")

# Step.7 消融分析包

把判準需要的數字**存成檔案**（不只是印在畫面上），打包成一個幾十 KB 的 zip。
跑完每個臂只要下載這一個檔就夠比對，不必傳整包 `runs_{ARM}.zip`。

| 檔案 | 內容 |
| --- | --- |
| `results.csv` | 逐輪指標 —— 主判準（平台期平均與 σ）的來源 |
| `args.yaml` | 該臂實際套用的超參數，用來核對沒有設定漂移 |
| `per_class.csv` | 每類 P / R / F1 / AP50 / AP50-95 —— 次判準 Thrips AP 在這裡 |
| `confusion_matrix.csv` | 混淆矩陣原始數字 —— 次判準 Scale_Insect FP 數在這裡 |
| `f1_conf.csv` | F1-信心曲線，用來找最佳截斷點 |
| `summary.json` | 平台期統計、速度、最佳 conf，一眼可讀 |

In [ ]:
import csv, json, os, shutil, zipfile

# 只用標準庫，不依賴 pandas —— ultralytics 本身也沒有強制要求它
def write_csv(path, fieldnames, rows):
    with open(path, "w", encoding="utf-8", newline="") as f:
        w = csv.DictWriter(f, fieldnames=fieldnames)
        w.writeheader()
        w.writerows(rows)


# 用 best.pt 重跑一次驗證，直接拿到 metrics 與混淆矩陣物件。
# 訓練結束時 Model.train() 已把 self.model 換成 best.pt，所以這裡驗的就是 best。
#
# plots=True 是必要的，不是為了畫圖：ultralytics 把 confusion_matrix.process_batch
# 包在 `if self.args.plots` 裡（detect/val.py:196），plots=False 會讓混淆矩陣
# 維持全零，Scale_Insect 的 FP 數就永遠是 0。
m = model.val(data=DATA_YAML, imgsz=HP["imgsz"], batch=HP["batch"], plots=True)

# 訓練輸出目錄：model.trainer 在 train() 之後仍在，save_dir 一定有效
RUN_DIR = str(model.trainer.save_dir)
OUT = f"/kaggle/working/ablation_{ARM}"
os.makedirs(OUT, exist_ok=True)

# ── 1. 逐輪指標與超參數（直接複製）──────────────────────────────────
for fn in ("results.csv", "args.yaml"):
    src = os.path.join(RUN_DIR, fn)
    if os.path.exists(src):
        shutil.copy2(src, os.path.join(OUT, fn))

# ── 2. 每類指標 ────────────────────────────────────────────────────
names = m.names if isinstance(m.names, dict) else {i: n for i, n in enumerate(m.names)}
COLS = ["class_id", "class", "precision", "recall", "f1", "ap50", "ap50_95"]
per_class = []
for i, ci in enumerate(m.box.ap_class_index):
    ci = int(ci)
    per_class.append({
        "class_id": ci,
        "class": names.get(ci, str(ci)),
        "precision": round(float(m.box.p[i]), 5),
        "recall": round(float(m.box.r[i]), 5),
        "f1": round(float(m.box.f1[i]), 5),
        "ap50": round(float(m.box.ap50[i]), 5),
        "ap50_95": round(float(m.box.ap[i]), 5),
    })
per_class.sort(key=lambda r: r["class_id"])
write_csv(os.path.join(OUT, "per_class.csv"), COLS, per_class)

# ── 3. 混淆矩陣原始數字（列=預測，欄=真實，最後一列/欄為背景）──────
cm = m.confusion_matrix.matrix
nc = len(names)
labels = [names.get(i, str(i)) for i in range(nc)] + ["background"]
with open(os.path.join(OUT, "confusion_matrix.csv"), "w", encoding="utf-8", newline="") as f:
    w = csv.writer(f)
    w.writerow([""] + [f"true_{l}" for l in labels])
    for i, lab in enumerate(labels):
        w.writerow([f"pred_{lab}"] + [int(cm[i][j]) for j in range(len(labels))])

# 每類的 FP / FN（背景那一列/欄）—— v8 報告裡 Scale_Insect 的 615 FP 就是這樣算的
fp_bg = {labels[i]: int(cm[i][nc]) for i in range(nc)}
fn_bg = {labels[i]: int(cm[nc][i]) for i in range(nc)}

# ── 4. F1-信心曲線與最佳截斷點 ──────────────────────────────────────
best_conf = None
try:
    x, y, _xl, _yl = m.curves_results[1]        # F1-Confidence(B)
    x = [float(v) for v in x]
    mean_f1 = ([sum(col) / len(col) for col in zip(*y)] if hasattr(y[0], "__len__")
               else [float(v) for v in y])
    with open(os.path.join(OUT, "f1_conf.csv"), "w", encoding="utf-8", newline="") as f:
        w = csv.writer(f)
        w.writerow(["conf", "mean_f1"])
        w.writerows(zip(x, mean_f1))
    best_conf = round(x[mean_f1.index(max(mean_f1))], 4)
except Exception as e:
    print(f"▷ F1-conf 曲線取用失敗（不影響主判準）：{type(e).__name__}: {e}")

# ── 5. 平台期統計 —— 這是主判準，直接算好省得事後再算 ────────────────
plateau = {}
rcsv = os.path.join(OUT, "results.csv")
if os.path.exists(rcsv):
    with open(rcsv, encoding="utf-8") as f:
        rec = [{k.strip(): v for k, v in row.items()} for row in csv.DictReader(f)]
    win = 50 if EPOCHS >= 120 else 16          # 長跑取最後 50 輪，短跑取最後 16 輪
    tail = rec[-win:]
    for key, col in (("mAP50", "metrics/mAP50(B)"), ("mAP50_95", "metrics/mAP50-95(B)")):
        if rec and col in rec[0]:
            vals = [float(r[col]) for r in tail]
            mean = sum(vals) / len(vals)
            var = sum((v - mean) ** 2 for v in vals) / len(vals)
            plateau[key] = {
                "window": f"last {len(vals)} epochs",
                "mean": round(mean, 5),
                "std": round(var ** 0.5, 5),
                "max_all_epochs": round(max(float(r[col]) for r in rec), 5),
            }
    if rec and "time" in rec[0]:
        plateau["s_per_epoch"] = round(float(rec[-1]["time"]) / len(rec), 1)

summary = {
    "arm": ARM,
    "desc": v9.ARMS[ARM]["desc"],
    "epochs": EPOCHS,
    "arch": v9.ARMS[ARM]["arch"],
    "loss_ratio": v9.ARMS[ARM]["loss_ratio"],
    "hyperparams": {k: (v if isinstance(v, (int, float, bool, type(None))) else str(v))
                    for k, v in sorted(HP.items())},
    "final_best_pt": {"mAP50": round(float(m.box.map50), 5),
                      "mAP50_95": round(float(m.box.map), 5),
                      "precision": round(float(m.box.mp), 5),
                      "recall": round(float(m.box.mr), 5)},
    "plateau": plateau,
    "best_f1_conf": best_conf,
    "fp_from_background": fp_bg,
    "fn_to_background": fn_bg,
    "speed_ms": {k: round(float(v), 3) for k, v in m.speed.items()},
}
with open(os.path.join(OUT, "summary.json"), "w", encoding="utf-8") as f:
    json.dump(summary, f, ensure_ascii=False, indent=2)

# ── 6. 打包成 Kaggle output 的 zip ──────────────────────────────────
ZIP = f"/kaggle/working/ablation_{ARM}.zip"
with zipfile.ZipFile(ZIP, "w", zipfile.ZIP_DEFLATED) as z:
    for fn in sorted(os.listdir(OUT)):
        z.write(os.path.join(OUT, fn), arcname=f"{ARM}/{fn}")

print("═" * 72)
print(f"▷ 分析包：{ZIP}  ({os.path.getsize(ZIP) / 1024:.1f} KB)")
print(f"   內含 {sorted(os.listdir(OUT))}")
print("═" * 72)
for k, v in plateau.items():
    if isinstance(v, dict):
        print(f"  {k:<10} 平台期({v['window']}) 平均={v['mean']:.5f}  σ={v['std']:.5f}"
              f"   全程最大={v['max_all_epochs']:.5f}")
    else:
        print(f"  {k:<10} {v}")
print(f"  最佳 F1 截斷點 conf = {best_conf}")

print(f"\n{'類別':<20}{'P':>9}{'R':>9}{'F1':>9}{'AP50':>9}{'AP50-95':>10}")
for r in per_class:
    print(f"{r['class']:<20}{r['precision']:>9.4f}{r['recall']:>9.4f}{r['f1']:>9.4f}"
          f"{r['ap50']:>9.4f}{r['ap50_95']:>10.4f}")

print(f"\n  背景誤報 FP：{fp_bg}")
print(f"  漏檢至背景 FN：{fn_bg}")
print(f"\n▷ 下載 ablation_{ARM}.zip 即可，不需要整包 runs_{ARM}.zip")